# Store Sales Feature Engineering and Model Evaluation

This notebook builds the 16-day forecasting model from cutoff-safe information.
Ridge provides the baseline, XGBoost provides the challenger, and validation
selects the configuration saved for repeatable batch inference.

**Inputs:**

- `data/processed/00_STORE_SALES_EDA.csv`
- `data/processed/01_STORE_SALES_KAGGLE_TEST.csv`
- `data/raw/sample_submission.csv`

**Generated private outputs:**

- `artifacts/store_sales_forecast_v1.pkl`
- `artifacts/store_sales_forecast_v1.json`
- `data/processed/02_STORE_SALES_KAGGLE_SUBMISSION.csv`
- `data/processed/03_STORE_SALES_BATCH_FORECAST.csv`

## Modeling sequence

1. Validate the processed interfaces.
2. Add exact 16, 21, 28, and 35-day sales lags.
3. Apply the fixed train, validation, and internal-test windows.
4. Fit one-hot encoding and numeric imputation on training rows only.
5. Compare Ridge and XGBoost on validation.
6. Refit the validation winner on train plus validation and evaluate the internal test once.
7. Diagnose the frozen internal-test errors without changing the selection.
8. Refit on all labels, save and reload the model artifact, then run the 16-day batch forecast.


In [ ]:
from pathlib import Path
import importlib
from time import perf_counter
import gc

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import ParameterGrid

import store_sales_model

store_sales_model = importlib.reload(store_sales_model)

from store_sales_model import (
    CATEGORICAL_FEATURES,
    DEFAULT_ARTIFACT_PATH,
    DEFAULT_BATCH_OUTPUT_PATH,
    MODEL_FEATURES,
    MODEL_GRIDS,
    NUMERIC_FEATURES,
    SALES_LAG_COLUMNS,
    SALES_LAGS,
    SALES_SUMMARY_COLUMNS,
    add_exact_sales_lags,
    build_model,
    evaluate_forecast,
    find_model_start,
    fit_forecast_bundle,
    is_default_reference,
    load_forecast_artifact,
    make_feature_processor,
    predict_forecast,
    save_forecast_artifact,
    write_batch_forecast,
)
from store_sales_preprocessing import (
    BASE_KEY,
    CALENDAR_COLUMNS,
    HOLIDAY_FEATURE_COLUMNS,
    OIL_AGE_COLUMNS,
    OIL_FEATURE_COLUMNS,
    OIL_LAG_COLUMNS,
    STORE_OUTPUT_COLUMNS,
    TEST_COLUMNS,
    TRAIN_COLUMNS,
    TRANSACTION_FEATURE_COLUMNS,
    TRANSACTION_LAG_COLUMNS,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

FIGURE_DPI = 150
FIGSIZE_16_9 = (12.8, 7.2)
FIGSIZE_4_3 = (10.0, 7.5)

METHOD_COLORS = {
    "Ridge Regression": "#D97706",
    "XGBoost Regression": "#2F6690",
}
ACTUAL_COLOR = "#2F6690"
FORECAST_COLOR = "#D97706"


def format_millions(value: float, _position: int) -> str:
    return f"{value / 1_000_000:,.1f}M"


LABELED_PATH = Path("data/processed/00_STORE_SALES_EDA.csv")
KAGGLE_TEST_PATH = Path("data/processed/01_STORE_SALES_KAGGLE_TEST.csv")
SAMPLE_SUBMISSION_PATH = Path("data/raw/sample_submission.csv")
SUBMISSION_PATH = Path("data/processed/02_STORE_SALES_KAGGLE_SUBMISSION.csv")
ARTIFACT_PATH = DEFAULT_ARTIFACT_PATH
BATCH_FORECAST_PATH = DEFAULT_BATCH_OUTPUT_PATH

required_paths = [LABELED_PATH, KAGGLE_TEST_PATH, SAMPLE_SUBMISSION_PATH]
missing_paths = [str(path) for path in required_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(
        "Required modeling files are missing: " + ", ".join(missing_paths)
        + ". Run 01_STORE_SALES_PREPROCESSING.ipynb first."
    )

print("Modeling environment configured successfully.")


## 1. Load and validate the processed interfaces

Match both saved headers to notebook 01 before loading the full tables.
`load_summary` shows the accepted shapes and date ranges after target checks.


In [ ]:
LABELED_COLUMNS = (
    TRAIN_COLUMNS
    + STORE_OUTPUT_COLUMNS
    + CALENDAR_COLUMNS
    + OIL_FEATURE_COLUMNS
    + TRANSACTION_FEATURE_COLUMNS
    + HOLIDAY_FEATURE_COLUMNS
)
KAGGLE_COLUMNS = (
    TEST_COLUMNS
    + STORE_OUTPUT_COLUMNS
    + CALENDAR_COLUMNS
    + OIL_FEATURE_COLUMNS
    + TRANSACTION_FEATURE_COLUMNS
    + HOLIDAY_FEATURE_COLUMNS
)

saved_labeled_columns = pd.read_csv(LABELED_PATH, nrows=0).columns.tolist()
saved_kaggle_columns = pd.read_csv(KAGGLE_TEST_PATH, nrows=0).columns.tolist()
if saved_labeled_columns != LABELED_COLUMNS or saved_kaggle_columns != KAGGLE_COLUMNS:
    raise ValueError(
        "The processed CSV headers do not match the final feature contract. "
        "Run 01_STORE_SALES_PREPROCESSING.ipynb from top to bottom before modeling."
    )

CATEGORY_COLUMNS = ["family", "city", "state", "store_type"]
INTEGER_COLUMNS = [
    "store_nbr",
    "store_cluster",
    *CALENDAR_COLUMNS,
    "transactions_lag_available_count",
    *HOLIDAY_FEATURE_COLUMNS,
]
FLOAT_COLUMNS = [
    *OIL_LAG_COLUMNS,
    *OIL_AGE_COLUMNS,
    *TRANSACTION_LAG_COLUMNS,
]
COMMON_DTYPES = {
    "id": "int32",
    "store_nbr": "int16",
    "onpromotion": "int32",
    **{column: "category" for column in CATEGORY_COLUMNS},
    **{column: "int16" for column in INTEGER_COLUMNS},
    **{column: "float32" for column in FLOAT_COLUMNS},
}

labeled = pd.read_csv(
    LABELED_PATH,
    dtype={**COMMON_DTYPES, "sales": "float32"},
    parse_dates=["date"],
)
kaggle_test = pd.read_csv(
    KAGGLE_TEST_PATH,
    dtype=COMMON_DTYPES,
    parse_dates=["date"],
)

assert not labeled.duplicated(BASE_KEY).any(), "The labeled base key is not unique."
assert not kaggle_test.duplicated(BASE_KEY).any(), "The Kaggle base key is not unique."
assert labeled["sales"].notna().all(), "The labeled target contains missing values."
assert (labeled["sales"] >= 0).all(), "The labeled target contains negative values."
assert "sales" not in kaggle_test.columns

load_summary = pd.DataFrame(
    {
        "dataset": ["Labeled history", "Kaggle inference"],
        "rows": [len(labeled), len(kaggle_test)],
        "first_date": [labeled["date"].min(), kaggle_test["date"].min()],
        "last_date": [labeled["date"].max(), kaggle_test["date"].max()],
        "columns_loaded": [labeled.shape[1], kaggle_test.shape[1]],
    }
)
display(load_summary)


## 2. Create exact historical sales features

The full horizon is forecast at once, so sales history begins at lag 16. Match
sales at 16, 21, 28, and 35 days by store and product family, then count the
available references for each row.


In [ ]:
labeled_features = add_exact_sales_lags(labeled, labeled)
kaggle_features = add_exact_sales_lags(kaggle_test, labeled)
del labeled, kaggle_test
gc.collect()

assert labeled_features.shape[1] == 39
assert kaggle_features.shape[1] == 38
print("Exact 16, 21, 28, and 35-day sales history is ready.")


## 3. Apply warm-up and fixed evaluation windows

Remove the initial warm-up rows, then apply the fixed chronological train,
validation, and internal-test boundaries. The three audit tables show population
sizes, horizon length, and lag coverage.


In [ ]:
TRAIN_END = pd.Timestamp("2017-07-14")
VALIDATION_START = pd.Timestamp("2017-07-15")
VALIDATION_END = pd.Timestamp("2017-07-30")
INTERNAL_TEST_START = pd.Timestamp("2017-07-31")
INTERNAL_TEST_END = pd.Timestamp("2017-08-15")
KAGGLE_START = pd.Timestamp("2017-08-16")
KAGGLE_END = pd.Timestamp("2017-08-31")

warmup_required_columns = SALES_LAG_COLUMNS + OIL_LAG_COLUMNS
MODEL_START = find_model_start(labeled_features)

warmup_mask = labeled_features["date"] < MODEL_START
train_mask = labeled_features["date"].between(MODEL_START, TRAIN_END)
validation_mask = labeled_features["date"].between(VALIDATION_START, VALIDATION_END)
internal_test_mask = labeled_features["date"].between(
    INTERNAL_TEST_START, INTERNAL_TEST_END
)
development_mask = labeled_features["date"].between(MODEL_START, VALIDATION_END)
all_model_labeled_mask = labeled_features["date"] >= MODEL_START

assert labeled_features.loc[
    labeled_features["date"].eq(MODEL_START), warmup_required_columns
].notna().all().all()
assert not (train_mask & validation_mask).any()
assert not (validation_mask & internal_test_mask).any()
assert (warmup_mask | train_mask | validation_mask | internal_test_mask).all()
assert kaggle_features["date"].min() == KAGGLE_START
assert kaggle_features["date"].max() == KAGGLE_END

split_audit = pd.DataFrame(
    [
        [
            "Warm-up excluded from modeling",
            warmup_mask.sum(),
            labeled_features.loc[warmup_mask, "date"].nunique(),
        ],
        ["Train", train_mask.sum(), labeled_features.loc[train_mask, "date"].nunique()],
        [
            "Validation",
            validation_mask.sum(),
            labeled_features.loc[validation_mask, "date"].nunique(),
        ],
        [
            "Internal test",
            internal_test_mask.sum(),
            labeled_features.loc[internal_test_mask, "date"].nunique(),
        ],
    ],
    columns=["period", "rows", "observed_dates"],
)
display(
    pd.Series(
        {"model_start": MODEL_START.date().isoformat()}, name="warm_up"
    ).to_frame()
)
display(split_audit)

window_contract = pd.DataFrame(
    [
        ["Validation", VALIDATION_START, VALIDATION_END, TRAIN_END],
        ["Internal test", INTERNAL_TEST_START, INTERNAL_TEST_END, VALIDATION_END],
        ["Kaggle inference", KAGGLE_START, KAGGLE_END, INTERNAL_TEST_END],
    ],
    columns=["period", "forecast_start", "forecast_end", "cutoff"],
)
window_contract["days"] = (
    window_contract["forecast_end"] - window_contract["forecast_start"]
).dt.days + 1
window_contract["latest_lag_source"] = (
    window_contract["forecast_end"] - pd.Timedelta(days=min(SALES_LAGS))
)
window_contract["all_lags_available_by_cutoff"] = (
    window_contract["latest_lag_source"] <= window_contract["cutoff"]
)
assert window_contract["days"].eq(16).all()
assert window_contract["all_lags_available_by_cutoff"].all()
display(window_contract)

history_columns = SALES_LAG_COLUMNS + TRANSACTION_LAG_COLUMNS + OIL_LAG_COLUMNS
split_labels = np.select(
    [warmup_mask, train_mask, validation_mask, internal_test_mask],
    ["Warm-up", "Train", "Validation", "Internal test"],
    default="Unassigned",
)
missing_history_audit = (
    labeled_features[history_columns]
    .isna()
    .assign(split=split_labels)
    .groupby("split", observed=True)[history_columns]
    .mean()
    .mul(100)
    .round(3)
)
display(missing_history_audit)


## 4. Freeze the model feature contract

The modeling table contains 36 inputs plus `id`, `date`, and `sales`. Nine
categorical fields use train-fitted one-hot encoding; 27 numeric fields use
train-median imputation, missing indicators, and sparse-safe scaling.


In [ ]:
FORBIDDEN_FEATURES = {
    "year",
    "quarter",
    "iso_week",
    "is_weekend",
    "oil_lag_mean",
    "oil_lag_std",
    "oil_lag_available_count",
    "transactions_lag_mean",
    "transactions_lag_std",
    "sales_lag_mean",
    "sales_lag_std",
    "calendar_event_count",
    "scheduled_event_count",
    "transferred_away_count",
    "is_event",
    "is_additional",
    "is_bridge",
    "is_transfer",
    "is_work_day",
}

assert len(CATEGORICAL_FEATURES) == 9
assert len(NUMERIC_FEATURES) == 27
assert len(MODEL_FEATURES) == 36
assert not (FORBIDDEN_FEATURES & set(labeled_features.columns))
assert set(MODEL_FEATURES).issubset(labeled_features.columns)
assert set(MODEL_FEATURES).issubset(kaggle_features.columns)


### Fit-ready feature transformation

`feature_contract` lists the categorical, numeric, traceability, and target
roles passed into the reusable transformer.


In [ ]:
feature_contract = pd.DataFrame(
    {
        "feature_group": ["Categorical", "Numeric", "Excluded", "Target"],
        "column_count_before_encoding": [9, 27, 2, 1],
        "handling": [
            "Train-fitted one-hot encoding",
            "Train median, missing indicators, and sparse-safe scaling",
            "id and date remain outside the model",
            "sales uses log1p during fitting",
        ],
    }
)
display(feature_contract)
print("Model-ready table: 39 visible columns = 36 inputs + id + date + sales")


## 5. Define the two-method tuning contract

Ridge is the regularized baseline; XGBoost is the nonlinear challenger. Each
declared parameter has three candidate values, including the default reference,
and every configuration uses the same chronological validation window.


In [ ]:
method_contract = pd.DataFrame(
    {
        "method": list(MODEL_GRIDS),
        "role": ["Simple ML baseline", "Mainstream challenger"],
        "configurations": [
            len(list(ParameterGrid(grid))) for grid in MODEL_GRIDS.values()
        ],
    }
)
assert method_contract["configurations"].tolist() == [3, 27]
display(method_contract)


### Evaluation and search evidence

Evaluate RMSLE, WAPE, signed bias, underforecast, and overforecast on the
original sales scale. `tuning_contract` lists the complete search before fitting.


In [ ]:
tuning_contract = pd.DataFrame(
    [
        {
            "method": method,
            "role": (
                "Simple ML baseline"
                if method == "Ridge Regression"
                else "Mainstream challenger"
            ),
            "tuned_parameters": ", ".join(grid),
            "values_per_parameter": 3,
            "total_runs": len(list(ParameterGrid(grid))),
            "includes_default_reference": True,
        }
        for method, grid in MODEL_GRIDS.items()
    ]
)
display(tuning_contract)


## 6. Tune both methods on validation

Fit the feature processor on training rows and reuse it unchanged for validation.
All models learn `log1p(sales)` and return non-negative sales forecasts. The full
30-run search stays in `validation_grid_results`; `method_summary` shows the best
configuration from each method.


In [ ]:
X_train = labeled_features.loc[train_mask, MODEL_FEATURES]
y_train = labeled_features.loc[train_mask, "sales"]
X_validation = labeled_features.loc[validation_mask, MODEL_FEATURES]
y_validation = labeled_features.loc[validation_mask, "sales"]

validation_processor = make_feature_processor()
X_train_matrix = validation_processor.fit_transform(X_train).astype(np.float32)
X_validation_matrix = validation_processor.transform(X_validation).astype(np.float32)
y_train_log = np.log1p(y_train.to_numpy(dtype=np.float32))

numeric_imputer = (
    validation_processor.named_transformers_["numeric"]
    .named_steps["imputer"]
)
indicator_indices = numeric_imputer.indicator_.features_
indicator_sources = [NUMERIC_FEATURES[index] for index in indicator_indices]
transformed_contract = pd.Series(
    {
        "rows_train": X_train_matrix.shape[0],
        "rows_validation": X_validation_matrix.shape[0],
        "model_columns_after_encoding_and_indicators": X_train_matrix.shape[1],
        "numeric_columns_with_missing_indicators": len(indicator_sources),
        "indicator_source_columns": ", ".join(indicator_sources),
    },
    name="value",
).to_frame()
display(transformed_contract)



### Validation search

Display the best Ridge and XGBoost rows used for the selection decision.


In [ ]:
validation_records = []
parameter_registry = {}
for method, grid in MODEL_GRIDS.items():
    configurations = list(ParameterGrid(grid))
    for run_number, parameters in enumerate(configurations, start=1):
        run_id = f"{method.split()[0].lower()}_{run_number:02d}"
        parameter_registry[run_id] = dict(parameters)
        model = build_model(method, parameters)

        started_at = perf_counter()
        model.fit(X_train_matrix, y_train_log)
        fit_seconds = perf_counter() - started_at

        predicted_log = model.predict(X_validation_matrix)
        predicted_sales = np.clip(np.expm1(predicted_log), 0.0, None)
        metrics = evaluate_forecast(y_validation, predicted_sales)
        validation_records.append(
            {
                "run_id": run_id,
                "method": method,
                "parameters": repr(dict(parameters)),
                "default_reference": is_default_reference(method, dict(parameters)),
                **metrics,
                "fit_seconds": fit_seconds,
            }
        )

validation_grid_results = pd.DataFrame(validation_records).sort_values(
    ["method", "rmsle", "wape_pct"],
    ignore_index=True,
)
method_summary = (
    validation_grid_results
    .sort_values(["rmsle", "wape_pct"], ignore_index=True)
    .groupby("method", as_index=False, sort=False)
    .head(1)
    .sort_values(["rmsle", "wape_pct"], ignore_index=True)
)

display(method_summary)



In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE_4_3, dpi=FIGURE_DPI)
ax.bar(
    method_summary["method"],
    method_summary["rmsle"],
    color=[METHOD_COLORS[method] for method in method_summary["method"]],
)
ax.set_title("Best Validation RMSLE by Method")
ax.set_ylabel("RMSLE (lower is better)")
ax.set_xlabel("")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 7. Freeze the selected method and parameters

Select the lowest validation RMSLE, using WAPE as the tie-breaker.
`selection_record` fixes the method and parameters before internal testing.


In [ ]:
selected_row = method_summary.iloc[0]
selected_method = str(selected_row["method"])
selected_run_id = str(selected_row["run_id"])
selected_parameters = parameter_registry[selected_run_id]

selection_record = method_summary.iloc[[0]].copy()
selection_record.insert(2, "decision", "Frozen before internal-test evaluation")
display(selection_record)
print(f"Selected method: {selected_method}")
print(f"Selected parameters: {selected_parameters}")

## 8. Evaluate the internal test once

Refit the selected configuration on train plus validation, then score the later
16-day internal test once. `internal_result` reports error, signed bias, forecast
direction, and fit time alongside the daily aggregate chart.


In [ ]:
del X_train, X_validation, X_train_matrix, X_validation_matrix, validation_processor
gc.collect()

X_development = labeled_features.loc[development_mask, MODEL_FEATURES]
y_development = labeled_features.loc[development_mask, "sales"]
X_internal_test = labeled_features.loc[internal_test_mask, MODEL_FEATURES]
y_internal_test = labeled_features.loc[internal_test_mask, "sales"]

internal_processor = make_feature_processor()
X_development_matrix = internal_processor.fit_transform(X_development).astype(np.float32)
X_internal_test_matrix = internal_processor.transform(X_internal_test).astype(np.float32)

internal_model = build_model(selected_method, selected_parameters)
internal_fit_started = perf_counter()
internal_model.fit(
    X_development_matrix,
    np.log1p(y_development.to_numpy(dtype=np.float32)),
)
internal_fit_seconds = perf_counter() - internal_fit_started
internal_prediction = np.clip(
    np.expm1(internal_model.predict(X_internal_test_matrix)),
    0.0,
    None,
)

internal_metrics = evaluate_forecast(y_internal_test, internal_prediction)
internal_result = pd.DataFrame(
    [
        {
            "method": selected_method,
            "parameters": repr(selected_parameters),
            "evaluation_period": "2017-07-31 to 2017-08-15",
            **internal_metrics,
            "fit_seconds": internal_fit_seconds,
        }
    ]
)
display(internal_result)

daily_internal = labeled_features.loc[internal_test_mask, ["date", "sales"]].copy()
daily_internal["predicted_sales"] = internal_prediction
daily_internal = daily_internal.groupby("date", as_index=False)[
    ["sales", "predicted_sales"]
].sum()



In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE_16_9, dpi=FIGURE_DPI)
ax.plot(
    daily_internal["date"],
    daily_internal["sales"],
    marker="o",
    color=ACTUAL_COLOR,
    label="Actual sales",
)
ax.plot(
    daily_internal["date"],
    daily_internal["predicted_sales"],
    marker="o",
    color=FORECAST_COLOR,
    label="Forecast sales",
)
ax.set_title("Internal Test: Daily Actual and Forecast Sales")
ax.yaxis.set_major_formatter(FuncFormatter(format_millions))
ax.set_ylabel("Total sales (millions)")
ax.set_xlabel("Date")
ax.legend()
ax.grid(alpha=0.25)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 9. Diagnose the frozen internal-test errors

Keep the selected model fixed and break its internal-test errors down by
forecast day, store, product family, promotion, and holiday status.

WAPE measures absolute error relative to actual sales, signed bias separates
overforecast from underforecast, and error contribution identifies the largest
shares of total absolute error.


In [ ]:
diagnostic_columns = [
    "date",
    "store_nbr",
    "family",
    "sales",
    "onpromotion",
    "is_holiday",
]
internal_diagnostics = labeled_features.loc[
    internal_test_mask,
    diagnostic_columns,
].copy()
internal_diagnostics["predicted_sales"] = internal_prediction.astype(np.float64)
internal_diagnostics["forecast_day"] = (
    internal_diagnostics["date"] - internal_diagnostics["date"].min()
).dt.days + 1
internal_diagnostics["promotion_segment"] = np.where(
    internal_diagnostics["onpromotion"].gt(0),
    "Promotion",
    "No promotion",
)
internal_diagnostics["holiday_segment"] = np.where(
    internal_diagnostics["is_holiday"].eq(1),
    "Holiday",
    "Non-holiday",
)
internal_diagnostics["absolute_error"] = (
    internal_diagnostics["predicted_sales"] - internal_diagnostics["sales"]
).abs()
internal_diagnostics["signed_error"] = (
    internal_diagnostics["predicted_sales"] - internal_diagnostics["sales"]
)
internal_diagnostics["squared_log_error"] = (
    np.log1p(internal_diagnostics["predicted_sales"])
    - np.log1p(internal_diagnostics["sales"])
) ** 2


def summarize_forecast_errors(
    frame: pd.DataFrame,
    group_columns: list[str],
) -> pd.DataFrame:
    summary = (
        frame.groupby(group_columns, observed=True, as_index=False)
        .agg(
            rows=("sales", "size"),
            actual_sales=("sales", "sum"),
            predicted_sales=("predicted_sales", "sum"),
            absolute_error=("absolute_error", "sum"),
            signed_error=("signed_error", "sum"),
            mean_squared_log_error=("squared_log_error", "mean"),
        )
    )
    denominator = summary["actual_sales"].replace(0, np.nan)
    summary["rmsle"] = np.sqrt(summary.pop("mean_squared_log_error"))
    summary["wape_pct"] = summary["absolute_error"] / denominator * 100
    summary["signed_bias_pct"] = summary.pop("signed_error") / denominator * 100
    summary["error_contribution_pct"] = (
        summary["absolute_error"] / internal_diagnostics["absolute_error"].sum() * 100
    )
    return summary


daily_error_summary = summarize_forecast_errors(
    internal_diagnostics,
    ["forecast_day", "date"],
).sort_values("forecast_day", ignore_index=True)
store_error_summary = summarize_forecast_errors(
    internal_diagnostics,
    ["store_nbr"],
).sort_values("absolute_error", ascending=False, ignore_index=True)
family_error_summary = summarize_forecast_errors(
    internal_diagnostics,
    ["family"],
).sort_values("absolute_error", ascending=False, ignore_index=True)
promotion_error_summary = summarize_forecast_errors(
    internal_diagnostics,
    ["promotion_segment"],
).sort_values("promotion_segment", ignore_index=True)
holiday_error_summary = summarize_forecast_errors(
    internal_diagnostics,
    ["holiday_segment"],
).sort_values("holiday_segment", ignore_index=True)

print("Daily error across the 16-day horizon")
display(daily_error_summary)
print("Stores contributing the most absolute error")
display(store_error_summary.head(10))
print("Product families contributing the most absolute error")
display(family_error_summary.head(10))
print("Promotion segments")
display(promotion_error_summary)
print("Holiday segments")
display(holiday_error_summary)



In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE_16_9, dpi=FIGURE_DPI)
ax.plot(
    daily_error_summary["forecast_day"],
    daily_error_summary["wape_pct"],
    marker="o",
    color=ACTUAL_COLOR,
)
ax.set_title("WAPE by Forecast Day")
ax.set_xlabel("Forecast day")
ax.set_ylabel("WAPE (%)")
ax.set_xticks(range(1, 17))
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

bias_colors = np.where(
    daily_error_summary["signed_bias_pct"].ge(0),
    FORECAST_COLOR,
    ACTUAL_COLOR,
)
fig, ax = plt.subplots(figsize=FIGSIZE_16_9, dpi=FIGURE_DPI)
ax.bar(
    daily_error_summary["forecast_day"],
    daily_error_summary["signed_bias_pct"],
    color=bias_colors,
)
ax.axhline(0, color="#475569", linewidth=1)
ax.set_title("Daily Bias: Positive Over, Negative Under")
ax.set_xlabel("Forecast day")
ax.set_ylabel("Signed bias (%)")
ax.set_xticks(range(1, 17))
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 10. Save the selected model and run reusable batch inference

Fit the selected configuration through 15 August 2017, save the processor and
model together, reload the artifact, and forecast the 16-31 August Kaggle rows.

The planning batch keeps date, store, and product-family identity. A separate
two-column file preserves Kaggle's required ID order.


In [ ]:
gc.collect()

evaluation_reference = {
    "validation_period": "2017-07-15 to 2017-07-30",
    "validation_rmsle": float(selected_row["rmsle"]),
    "validation_wape_pct": float(selected_row["wape_pct"]),
    "validation_signed_bias_pct": float(selected_row["signed_bias_pct"]),
    "internal_test_period": "2017-07-31 to 2017-08-15",
    "internal_test_rmsle": float(internal_metrics["rmsle"]),
    "internal_test_wape_pct": float(internal_metrics["wape_pct"]),
    "internal_test_signed_bias_pct": float(internal_metrics["signed_bias_pct"]),
    "evidence_scope": "Controlled local evaluation; not live retail validation",
}

final_training_data = labeled_features.loc[
    all_model_labeled_mask,
    ["date", "sales", *MODEL_FEATURES],
]
final_bundle = fit_forecast_bundle(
    final_training_data,
    selected_method,
    selected_parameters,
    evaluation_reference,
)
artifact_path, metadata_path = save_forecast_artifact(
    final_bundle,
    ARTIFACT_PATH,
    overwrite=True,
)

reloaded_bundle = load_forecast_artifact(artifact_path)
batch_forecast = predict_forecast(reloaded_bundle, kaggle_features)
write_batch_forecast(
    batch_forecast,
    BATCH_FORECAST_PATH,
    overwrite=True,
)
kaggle_prediction = batch_forecast["forecast_sales"].to_numpy()

sample_submission = pd.read_csv(
    SAMPLE_SUBMISSION_PATH,
    dtype={"id": "int64", "sales": "float32"},
)
assert len(sample_submission) == len(kaggle_features)
assert np.array_equal(
    sample_submission["id"].to_numpy(),
    batch_forecast["id"].to_numpy(),
)
assert len(kaggle_prediction) == len(sample_submission)
assert np.isfinite(kaggle_prediction).all()
assert (kaggle_prediction >= 0).all()

submission = sample_submission.copy()
submission["sales"] = kaggle_prediction.astype(np.float32)
submission.to_csv(SUBMISSION_PATH, index=False)

artifact_summary = pd.Series(
    {
        "model_version": reloaded_bundle["metadata"]["model_version"],
        "artifact_file": artifact_path.name,
        "metadata_file": metadata_path.name,
        "training_end": reloaded_bundle["metadata"]["training_end"],
        "training_rows": reloaded_bundle["metadata"]["training_rows"],
        "transformed_features": reloaded_bundle["metadata"][
            "transformed_feature_count"
        ],
        "batch_rows": len(batch_forecast),
        "forecast_start": batch_forecast["date"].min(),
        "forecast_end": batch_forecast["date"].max(),
        "minimum_prediction": batch_forecast["forecast_sales"].min(),
        "mean_prediction": batch_forecast["forecast_sales"].mean(),
        "maximum_prediction": batch_forecast["forecast_sales"].max(),
        "id_order_matches_sample": True,
    },
    name="value",
).to_frame()
display(artifact_summary)
print(f"Private model artifact written to: {artifact_path}")
print(f"Private batch forecast written to: {BATCH_FORECAST_PATH}")
print(f"Private Kaggle submission written to: {SUBMISSION_PATH}")


## Modeling result

A complete run records every validation result, the selected method and
parameters, one internal-test evaluation, segmented error analysis,
and predictions from the reloaded model artifact. These outputs form the
reproducible v1 evidence.
